# Uni-Agent + veRL on a Colab A100 (no sandbox)

**Warning:** coding commands run directly in this Colab VM. Do not mount Google Drive or load secrets while rollouts are active. The agent can access or destroy the model, checkpoints, and any host files. It can also inspect verifier data in the repository or Parquet files, so this mode is not a secure benchmark and may permit reward hacking.

Upload or clone the customized repository to `/content/uniagent`, then select **Runtime → Change runtime type → A100 GPU**.

The preparation command below also reconstructs the base Parquet files when a GitHub clone omitted them because of the repository's global `*.parquet` ignore rule.

In [ ]:
import subprocess
from pathlib import Path

REPO = Path("/content/uniagent")
assert (REPO / "uni_agent").is_dir(), "Put this repository at /content/uniagent first"
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
print("GPU:", gpu)
assert "A100" in gpu, f"Select an A100 Colab runtime; found {gpu}"

In [ ]:
%cd /content/uniagent
!bash examples/mixed_code_react/setup_colab_a100.sh

Restart the Colab runtime now, then continue below. The files in `/content` remain, while Python reloads the newly installed PyTorch/vLLM stack.

In [ ]:
%cd /content/uniagent
!python examples/mixed_code_react/prepare_colab_host_data.py \
  --repo-root /content/uniagent \
  --node-modules /content/uniagent-react-runtime/node_modules

In [ ]:
%cd /content/uniagent
!MODEL_PATH=Qwen/Qwen3-0.6B bash examples/mixed_code_react/train_colab_a100.sh

Checkpoints are saved under `/content/uniagent-checkpoints`. Stop training before mounting Drive and copying them. Ngrok is intentionally absent because the agent and model share this VM.